In [5]:
import pandas as pd
import duckdb
import unicodedata
import os
import warnings
from pathlib import Path

# Suprime o aviso inofensivo do openpyxl sobre a formatação do IBGE
warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

def padronizar_uf(nome):
    """
    Remove acentos e converte para snake_case (ex: 'São Paulo' -> 'sao_paulo')
    para garantir compatibilidade exata com as chaves categoriais do Parquet.
    """
    nome = unicodedata.normalize('NFKD', nome).encode('ASCII', 'ignore').decode('utf-8')
    return nome.lower().replace(' ', '_')

def extrair_totais_anuais(arquivos):
    dfs = []
    regioes_ibge = ['Brasil', 'Norte', 'Nordeste', 'Sudeste', 'Sul', 'Centro-Oeste']
    
    for arquivo, ano in arquivos.items():
        df = pd.read_excel(arquivo)
        
        df_clean = df.iloc[5:, [0, 3]].dropna().copy()
        df_clean.columns = ['uf_nome', 'quantidade_tons_ano']
        df_clean['uf_nome'] = df_clean['uf_nome'].astype(str).str.strip()
        
        df_estados = df_clean[~df_clean['uf_nome'].isin(regioes_ibge)].copy()
        df_estados['uf'] = df_estados['uf_nome'].apply(padronizar_uf)
        df_estados['ano'] = ano
        
        dfs.append(df_estados[['ano', 'uf', 'quantidade_tons_ano']])
        
    return pd.concat(dfs, ignore_index=True)

# 1. Definição do diretório base
caminho_raiz = Path.cwd().resolve()
while not (caminho_raiz / "data").exists() and caminho_raiz.name != "":
    caminho_raiz = caminho_raiz.parent

caminho_base = caminho_raiz / "data" / "raw" / "safra" 

arquivos_excel = {
    caminho_base / '2004-tab02_feijao.xlsx': 2004,
    caminho_base / '2005-tab02_Feijao.xlsx': 2005,

}

df_historico_anual = extrair_totais_anuais(arquivos_excel)

# 2. Leitura do Parquet
caminho_parquet_path = caminho_raiz / "data" / "interim" / "safra" / "producao_feijao_tons.parquet"

# Ao carregar para o Pandas, chamamos a variável de df_parquet 
df_parquet = pd.read_parquet(caminho_parquet_path)

# AJUSTE CRÍTICO: Troca de 'estado_origem' para 'uf' e 'soma_mensal_producao' para 'quantidade_tons'
query = """
WITH producao_mensal_ref AS (
    SELECT uf, mes, SUM(quantidade_tons) AS prod_mes
    FROM df_parquet WHERE ano BETWEEN 2007 AND 2009 GROUP BY uf, mes
),
producao_anual_ref AS (
    SELECT uf, SUM(quantidade_tons) AS prod_ano
    FROM df_parquet WHERE ano BETWEEN 2007 AND 2009 GROUP BY uf
),
pesos_sazonais AS (
    SELECT m.uf, m.mes, m.prod_mes / NULLIF(a.prod_ano, 0) AS peso_mes
    FROM producao_mensal_ref m JOIN producao_anual_ref a ON m.uf = a.uf
)
SELECT 
    h.ano, p.mes, h.uf,
    ROUND(h.quantidade_tons_ano * p.peso_mes, 0) AS quantidade_tons,
    TRUE AS is_estimado
FROM df_historico_anual h
JOIN pesos_sazonais p ON h.uf = p.uf
WHERE h.ano = 2004 OR (h.ano = 2005 AND p.mes <= 10)
ORDER BY h.ano, p.mes, h.uf
"""

# 3. Execução do cálculo
df_estimado_2004_2006 = duckdb.sql(query).df()

# 4. Exportação do arquivo tratado 
caminho_saida = caminho_base / 'producao_mensal_estimada_2004_2006.parquet'
df_estimado_2004_2006.to_parquet(caminho_saida)

caminho_saida_xlsx = caminho_base / 'producao_mensal_estimada_2004_2006.xlsx'
df_estimado_2004_2006.to_excel(caminho_saida_xlsx, index=False)

print(f"Desagregação concluída. Base de origens salva em: {caminho_saida}")

Desagregação concluída. Base de origens salva em: C:\Users\leand\Documents\mecai-26-prob-estat\data\raw\safra\producao_mensal_estimada_2004_2006.parquet
